# 04_pipeline — End-to-End Clinical Decision Support

Wires the full 5-step pipeline into a single `predict(patient_row)` function:

1. Render patient note from MIMIC features.
2. BioMistral + **Stage 1 adapter** → parse `icd_titles`, `retrieval_query`, `adm_medications`.
3. `retrieve(query, k=5)` → top-5 CREST guidelines via FAISS.
4. Build Stage 2 prompt (patient note + ICD titles + retrieved snippets).
5. BioMistral + **Stage 2 adapter** → recommendation with inline `— Developer (strength)` citations.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q \
    transformers==4.47.0 \
    faiss-cpu \
    peft==0.12.0 \
    trl==0.10.1 \
    bitsandbytes>=0.46.1 \
    accelerate==0.34.2 \
    datasets==2.21.0 \
    sentence-transformers \
    python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.6.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
import os
import pickle
from kaggle_secrets import UserSecretsClient
import huggingface_hub

# Paths
WORKING_DIR = "/kaggle/working"
os.makedirs(WORKING_DIR, exist_ok=True)

# HuggingFace login
secrets = UserSecretsClient()
huggingface_hub.login(
    token=secrets.get_secret("HF_TOKEN"),
    add_to_git_credential=False
)

STAGE1_ADAPTER_PATH = "/kaggle/input/notebooks/aanyagupta0921/finetune-stage1-02/stage1_adapter"
STAGE2_ADAPTER_PATH = "/kaggle/input/notebooks/aanyagupta0921/finetune-stage2-03/stage2_adapter"
FAISS_INDEX_PATH    = "/kaggle/input/notebooks/aanyagupta0921/rag-index-01/crest.index"
ID_MAP_PATH         = "/kaggle/input/notebooks/aanyagupta0921/rag-index-01/crest_id_map.pkl"


In [3]:
import sys
sys.path.append("/kaggle/input/datasets/aanyagupta0921/senior-project-utils")

from utils import (
    load_biomistral_4bit,
    format_stage1_prompt,
    format_stage2_prompt,
    build_rag_index,
    retrieve,
    render_patient_note,
    _STAGE2_INSTRUCTION
)

In [4]:
import ast
import pandas as pd

mimic_df = pd.read_csv('/kaggle/input/datasets/aanyagupta0921/senior-project-mimic-dataset/mimic_data.csv')

LIST_COLS = ["icd_title", "med_record", "adm_med_name", "chiefcomplaint", "etcdescription"]
for col in LIST_COLS:
    mimic_df[col] = mimic_df[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
print(f"Loaded {len(mimic_df)} MIMIC rows for demo")


Loaded 126 MIMIC rows for demo


In [5]:
import faiss
import torch
from peft import PeftModel

# Base model (shared across both adapters)
base_model, tokenizer = load_biomistral_4bit()

# Load Stage 1 adapter, then add Stage 2 as a second named adapter
model = PeftModel.from_pretrained(base_model, STAGE1_ADAPTER_PATH, adapter_name="stage1")
model.load_adapter(STAGE2_ADAPTER_PATH, adapter_name="stage2")
model.eval()
print("Both adapters loaded.")

# FAISS index + id_map
faiss_index = faiss.read_index(FAISS_INDEX_PATH)
with open(ID_MAP_PATH, "rb") as f:
    id_map = pickle.load(f)
print(f"FAISS index: {faiss_index.ntotal} vectors  |  id_map: {len(id_map)} entries")


2026-05-08 22:24:49.370714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778279089.704824      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778279089.762999      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778279090.314966      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778279090.315010      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778279090.315013      57 computation_placer.cc:177] computation placer alr

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Both adapters loaded.
FAISS index: 1604 vectors  |  id_map: 1604 entries


In [6]:
import json as _json
import re as _re

TOP_K = 5


def _parse_stage1_output(raw):
    """Parse Stage 1 JSON output; falls back to regex on malformed responses."""
    clean = raw.split("</s>")[0].strip()
    try:
        parsed = _json.loads(clean)
        return (
            parsed.get("icd_titles", []),
            parsed.get("retrieval_query", "clinical guidelines"),
            parsed.get("adm_medications", []),
        )
    except _json.JSONDecodeError:
        pass
    icd_m   = _re.search(r'"icd_titles"\s*:\s*(\[.*?\])', clean, _re.DOTALL)
    query_m = _re.search(r'"retrieval_query"\s*:\s*"([^"]+)"', clean)
    meds_m  = _re.search(r'"adm_medications"\s*:\s*(\[.*?\])', clean, _re.DOTALL)
    return (
        _json.loads(icd_m.group(1))   if icd_m   else [],
        query_m.group(1)               if query_m else "clinical guidelines",
        _json.loads(meds_m.group(1))  if meds_m  else [],
    )


def _generate(prompt, max_new_tokens, do_sample, temperature=1.0):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


def predict(patient_row):
    """Run the full 5-step pipeline on a single MIMIC row."""
    # Step 1: patient note
    patient_note = render_patient_note(patient_row)

    # Step 2: Stage 1 → ICD + medications + retrieval query
    model.set_adapter("stage1")
    stage1_raw = _generate(
        format_stage1_prompt(patient_note),
        max_new_tokens=300, do_sample=False,
    )
    icd_titles, retrieval_query, adm_medications = _parse_stage1_output(stage1_raw)

    # Step 3: retrieve top-k CREST snippets
    snippets = retrieve(retrieval_query, faiss_index, id_map, k=TOP_K)

    # Step 4 + 5: Stage 2 → recommendation with citations
    model.set_adapter("stage2")
    recommendation = _generate(
        format_stage2_prompt(patient_note, icd_titles, snippets),
        max_new_tokens=512, do_sample=True, temperature=0.7,
    )

    return {
        "patient_note":      patient_note,
        "stage1_raw":        stage1_raw,
        "icd_titles":        icd_titles,
        "retrieval_query":   retrieval_query,
        "adm_medications":   adm_medications,
        "retrieved_snippets": snippets,
        "recommendation":    recommendation,
    }


print("predict() ready.")


predict() ready.


In [7]:
def display_trace(result, ground_truth_row=None):
    W = 72
    print("=" * W)
    print("PATIENT NOTE")
    print("-" * W)
    print(result["patient_note"])

    print()
    print("=" * W)
    print("STAGE 1  —  Predicted ICD Titles / Medications / Query")
    print("-" * W)
    print(f"ICD titles:      {result['icd_titles']}")
    print(f"Adm. meds:       {result['adm_medications']}")
    print(f"Retrieval query: {result['retrieval_query']}")

    if ground_truth_row is not None:
        print()
        print(f"[Ground truth] ICD:  {ground_truth_row['icd_title']}")
        print(f"[Ground truth] Meds: {ground_truth_row['adm_med_name']}")

    print()
    print("=" * W)
    print(f"RETRIEVED GUIDELINES  (top {len(result['retrieved_snippets'])})")
    print("-" * W)
    for i, s in enumerate(result["retrieved_snippets"], 1):
        title_short = s["title"][:60]
        dev_short   = s["developer"][:35]
        print(f"  {i}. [{s['recommendation_norm']:6s}] {title_short}... ({dev_short})")

    print()
    print("=" * W)
    print("FINAL RECOMMENDATION")
    print("-" * W)
    print(result["recommendation"])
    print("=" * W)


In [8]:
demo_rows = mimic_df.sample(n=5, random_state=7).reset_index(drop=True)

for i, (_, row) in enumerate(demo_rows.iterrows()):
    print(f"\n{'#' * 72}")
    print(f"  DEMO PATIENT {i+1} / {len(demo_rows)}  (stay_id={row['stay_id']})")
    print(f"{'#' * 72}\n")
    result = predict(row)
    display_trace(result, ground_truth_row=row)



########################################################################
  DEMO PATIENT 1 / 5  (stay_id=39300221)
########################################################################



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

PATIENT NOTE
------------------------------------------------------------------------
Patient Record
- Gender: FEMALE
- Race: WHITE
- Disposition: HOME
- Chief complaint: PICC LINE INFECTION
- Acuity: 3
- Pain score: 0
- Medication history: docusate sodium, oxycodone, ceftriaxone, Mapap (acetaminophen), metronidazole, levetiracetam

Vitals on Arrival
- Temperature: 98.0 F
- Heart rate: 75.0 bpm
- Respiratory rate: 20.0 breaths/min
- O2 saturation: 98.0%
- Blood pressure: 132.0/95.0 mmHg

ICU Vital Trends (mean, trend)
- Heart rate: 70.0 bpm, stable
- Respiratory rate: 14.0, stable
- O2 saturation: 99.0%, stable
- SBP: 130.0, stable
- DBP: 82.0, stable

STAGE 1  —  Predicted ICD Titles / Medications / Query
------------------------------------------------------------------------
ICD titles:      ['PICC LINE INFECTION']
Adm. meds:       ['Vancomycin', 'Ceftriaxone']
Retrieval query: guidelines for picc line infection

[Ground truth] ICD:  ['REMOVAL VASCULAR CATHETER']
[Ground truth] Meds

In [9]:
!pip install -q fastapi "uvicorn[standard]" pyngrok
import threading
import pandas as pd
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [10]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
token = secrets.get_secret("NGROK_AUTH_TOKEN")
print(f"Token found: {bool(token)}")  # should print True
print(f"Token preview: {token[:8]}...")  # should show first 8 chars
os.environ["NGROK_AUTH_TOKEN"] = token
ngrok.set_auth_token(token)  # set it explicitly before connect()

Token found: True
Token preview: 2kU6ooXU...


In [11]:
# copy the printed ngrok URL into your local .env as NGROK_URL,
# then run  scripts/start-mac.sh  to start the Docker frontend.

# Map the form's trend dropdown values to numerics that categorize_trend() reads correctly
_TREND_MAP = {"increasing": 1.0, "stable": 0.0, "decreasing": -1.0}

class _PatientInput(BaseModel):
    gender: str
    race: str
    disposition: str
    acuity: int
    pain: int
    chiefcomplaint: str   # comma-separated string
    med_record: str       # comma-separated string
    temperature: float
    heartrate: float
    resprate: float
    o2sat: float
    sbp: float
    dbp: float
    heartrate_vital_mean: float
    heartrate_vital_trend: str  # "increasing" | "stable" | "decreasing"
    resprate_vital_mean: float
    resprate_vital_trend: str
    o2sat_vital_mean: float
    o2sat_vital_trend: str
    sbp_vital_mean: float
    sbp_vital_trend: str
    dbp_vital_mean: float
    dbp_vital_trend: str


_server_app = FastAPI()
_server_app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_methods=["POST"], allow_headers=["*"],
)


@_server_app.post("/predict")
def _run_predict(inp: _PatientInput):
    row = pd.Series({
        "gender":     inp.gender,
        "race":       inp.race,
        "disposition": inp.disposition,
        "acuity":     inp.acuity,
        "pain":       inp.pain,
        "chiefcomplaint": [c.strip() for c in inp.chiefcomplaint.split(",") if c.strip()],
        "med_record":     [m.strip() for m in inp.med_record.split(",") if m.strip()],
        "temperature": inp.temperature,
        "heartrate":   inp.heartrate,
        "resprate":    inp.resprate,
        "o2sat":       inp.o2sat,
        "sbp":         inp.sbp,
        "dbp":         inp.dbp,
        "heartrate_vital_mean":  inp.heartrate_vital_mean,
        "heartrate_vital_trend": _TREND_MAP[inp.heartrate_vital_trend],
        "resprate_vital_mean":   inp.resprate_vital_mean,
        "resprate_vital_trend":  _TREND_MAP[inp.resprate_vital_trend],
        "o2sat_vital_mean":      inp.o2sat_vital_mean,
        "o2sat_vital_trend":     _TREND_MAP[inp.o2sat_vital_trend],
        "sbp_vital_mean":        inp.sbp_vital_mean,
        "sbp_vital_trend":       _TREND_MAP[inp.sbp_vital_trend],
        "dbp_vital_mean":        inp.dbp_vital_mean,
        "dbp_vital_trend":       _TREND_MAP[inp.dbp_vital_trend],
    })
    result = predict(row)
    return {
        "icd_titles":      result["icd_titles"],
        "adm_medications": result["adm_medications"],
        "recommendation":  result["recommendation"],
    }


def _start_server():
    uvicorn.run(_server_app, host="0.0.0.0", port=8001, log_level="error")


threading.Thread(target=_start_server, daemon=True).start()
tunnel = ngrok.connect(8001)
print(f"Ngrok URL: {tunnel.public_url}")
print()
print("Next step: add this line to your local .env, then run  scripts/start-mac.sh")
print(f"  NGROK_URL={tunnel.public_url}")


Ngrok URL: https://809e-34-138-89-244.ngrok-free.app

Next step: add this line to your local .env, then run  scripts/start-mac.sh
  NGROK_URL=https://809e-34-138-89-244.ngrok-free.app
